# **02. Data Cleaning — очистка и подготовка данных**

В этом ноутбуке сделаем второй этап проекта по датасету **The Movies Dataset** — очистка и подготовка данных после первичного аудита

На предыдущем этапе мы изучили исходные CSV-файлы, проверили размеры таблиц, типы данных, пропуски, дубликаты и ключи для объединения. По итогам аудита стало понятно, что данные можно использовать для анализа, но перед этим их нужно привести к более удобному и надёжному виду

Основная цель этого ноутбука — подготовить очищенные таблицы, которые дальше можно будет загрузить в PostgreSQL и использовать для создания аналитических витрин

Что мы сделаем?

- приведём типы данных к корректному формату
- обработаем некорректные значения в ключевых колонках
- удалим или обработаем дубликаты
- преобразуем даты и выделим год выпуска фильма
- обработаем числовые признаки, такие как бюджет, выручка, популярность и длительность
- распарсим сложные текстовые поля `genres`, `cast`, `crew` и `keywords`
- выделим основной жанр, режиссёра, размер актёрского состава и количество ключевых слов
- подготовим пользовательские рейтинги;
- объединим метаданные фильмов с рейтингами через таблицу связей
- сохраним очищенные данные в папку `data/processed`

Важно, чтобы данные были пригодны для дальнейшей работы в SQL, поэтому результатом ноутбука будут аккуратные таблицы с понятными названиями колонок, корректными типами данных и минимальным количеством технических проблем


----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [1]:
import pandas as pd
import numpy as np
from ast import literal_eval
from pathlib import Path

In [2]:
movies = pd.read_csv("../data/raw/movies_metadata.csv", low_memory=False)
ratings = pd.read_csv("../data/raw/ratings_small.csv")
credits = pd.read_csv("../data/raw/credits.csv")
keywords = pd.read_csv("../data/raw/keywords.csv")
links = pd.read_csv("../data/raw/links_small.csv")

In [3]:
#создаём папку и путь к ней, куда потом будем сохранять очищенные данные
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# 1) Очищаем movies

In [4]:
#убираем те самые битые 3 строки, которые выявили в прошлом ноутбуке
#приводим к числовым данным
movies["id_num"] = pd.to_numeric(movies["id"], errors="coerce")
movies = movies[movies["id_num"].notna()].copy()
movies["id_num"] = movies["id_num"].astype("int64")

In [5]:
movies["budget"] = pd.to_numeric(movies["budget"], errors="coerce")
movies["popularity"] = pd.to_numeric(movies["popularity"], errors="coerce")
movies["revenue"] = pd.to_numeric(movies["revenue"], errors="coerce")
movies["runtime"] = pd.to_numeric(movies["runtime"], errors="coerce")
movies["vote_average"] = pd.to_numeric(movies["vote_average"], errors="coerce")
movies["vote_count"] = pd.to_numeric(movies["vote_count"], errors="coerce")

movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")
movies["release_year"] = movies["release_date"].dt.year

In [6]:
movies[[
    "id",
    "id_num",
    "budget",
    "popularity",
    "revenue",
    "runtime",
    "vote_average",
    "vote_count",
    "release_date",
    "release_year"
]].head()

,id,id_num,budget,popularity,revenue,runtime,vote_average,vote_count,release_date,release_year
0,862,862,30000000,21.946943,373554033.0,81.0,7.7,5415.0,1995-10-30,1995.0
1,8844,8844,65000000,17.015539,262797249.0,104.0,6.9,2413.0,1995-12-15,1995.0
2,15602,15602,0,11.712900,0.0,101.0,6.5,92.0,1995-12-22,1995.0
3,31357,31357,16000000,3.859495,81452156.0,127.0,6.1,34.0,1995-12-22,1995.0
4,11862,11862,0,8.387519,76578911.0,106.0,5.7,173.0,1995-02-10,1995.0


In [7]:
#проверяем дубли в movies
#keep=False: если бы стояло по умолчанию, pandas пометил бы только повторные строки
#а с keep=False он помечает все строки из группы дублей, и первую, и вторую, и все остальные
movies[movies.duplicated("id_num", keep=False)].sort_values("id_num")[["id_num", "title", "release_date"]].head(20)

,id_num,title,release_date
5865,4912,Confessions of a Dangerous Mind,2002-12-30
33826,4912,Confessions of a Dangerous Mind,2002-12-30
9165,5511,Le Samouraï,1967-10-25
7345,5511,Le Samouraï,1967-10-25
44821,10991,Pokémon: Spell of the Unknown,2000-07-08
4114,10991,Pokémon: Spell of the Unknown,2000-07-08
14012,11115,Deal,2008-01-29
24844,11115,Deal,2008-01-29
44826,12600,Pokémon 4Ever: Celebi - Voice of the Forest,2001-07-06
5535,12600,Pokémon 4Ever: Celebi - Voice of the Forest,2001-07-06


In [8]:
#обычные технические дубли, т.к. полностью совпадают названия и даты релизов
#поэтому оставляем первую запись, а оставльные дубли удаляем
movies = movies.drop_duplicates(subset="id_num", keep="first").copy()

In [9]:
#проверим
print("Дубликаты id_num после очистки:", movies["id_num"].duplicated().sum())
print("Размер таблицы после удаления дублей:", movies.shape)

Дубликаты id_num после очистки: 0
Размер таблицы после удаления дублей: (45433, 26)


----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# 2) распарсим жанры

нам нужны: список жанров, количество жанров, основной жанр

In [10]:
#создадим функции, чтобы превращать такие строки в обычные списки
from ast import literal_eval 
import numpy as np


def safe_literal_eval(value): #превращается сложную строку в список
#если строка пустая, кривая или внутри ошибка, то вернётся пустой список
    if pd.isna(value):
        return []
    try:
        return literal_eval(value)
    except (ValueError, SyntaxError):
        return []
    

def extract_name_list(value): #достаёт из списка словарей только поле name
    items = safe_literal_eval(value)
    if isinstance(items, list):
        return [item.get("name") for item in items if isinstance(item, dict) and "name" in item]
    return []

In [11]:
#создаём новый столбец со списком жанров
movies["genres_list"] = movies["genres"].apply(extract_name_list)
#считаем, сколько у фильма жанров
movies["genre_count"] = movies["genres_list"].apply(len)
#берём первый жанр из списка (основной)
movies["main_genre"] = movies["genres_list"].apply(lambda x: x[0] if len(x) > 0 else np.nan)

In [12]:
#проверим результат
movies[["title", "genres", "genres_list", "genre_count", "main_genre"]].head(10)

,title,genres,genres_list,genre_count,main_genre
0,Toy Story,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[Animation, Comedy, Family]",3,Animation
1,Jumanji,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[Adventure, Fantasy, Family]",3,Adventure
2,Grumpier Old Men,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[Romance, Comedy]",2,Romance
3,Waiting to Exhale,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...","[Comedy, Drama, Romance]",3,Comedy
4,Father of the Bride Part II,"[{'id': 35, 'name': 'Comedy'}]",[Comedy],1,Comedy
5,Heat,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam...","[Action, Crime, Drama, Thriller]",4,Action
6,Sabrina,"[{'id': 35, 'name': 'Comedy'}, {'id': 10749, '...","[Comedy, Romance]",2,Comedy
7,Tom and Huck,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[Action, Adventure, Drama, Family]",4,Action
8,Sudden Death,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[Action, Adventure, Thriller]",3,Action
9,GoldenEye,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...","[Adventure, Action, Thriller]",3,Adventure


----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# 3) распарсим credits

нам нужны: режиссёры из crew, список первых актёров из cast, размер каста

In [13]:
#создадим функции, чтобы безопасно превращать такие строки в обычные списки
def get_director(value): #проходит по списку crew и ищет человека, у которого job == "Director"
    crew_list = safe_literal_eval(value)
    if isinstance(crew_list, list):
        for item in crew_list:
            if isinstance(item, dict) and item.get("job") == "Director":
                return item.get("name")
    return np.nan


def get_cast_names(value, top_n=3): #берёт список актёров из cast и оставляет первых трёх
    cast_list = safe_literal_eval(value)
    if isinstance(cast_list, list):
        names = [item.get("name") for item in cast_list if isinstance(item, dict) and "name" in item]
        return names[:top_n]
    return []

In [14]:
#список первых трёх актёров
credits["cast_list"] = credits["cast"].apply(get_cast_names)
#сколько всего актёров в касте
credits["cast_size"] = credits["cast"].apply(lambda x: len(safe_literal_eval(x)))
#имя режиссёра
credits["director"] = credits["crew"].apply(get_director)

In [15]:
#проверим результат
credits[["id", "cast_list", "cast_size", "director"]].head(10)

,id,cast_list,cast_size,director
0,862,"[Tom Hanks, Tim Allen, Don Rickles]",13,John Lasseter
1,8844,"[Robin Williams, Jonathan Hyde, Kirsten Dunst]",26,Joe Johnston
2,15602,"[Walter Matthau, Jack Lemmon, Ann-Margret]",7,Howard Deutch
3,31357,"[Whitney Houston, Angela Bassett, Loretta Devine]",10,Forest Whitaker
4,11862,"[Steve Martin, Diane Keaton, Martin Short]",12,Charles Shyer
5,949,"[Al Pacino, Robert De Niro, Val Kilmer]",65,Michael Mann
6,11860,"[Harrison Ford, Julia Ormond, Greg Kinnear]",57,Sydney Pollack
7,45325,"[Jonathan Taylor Thomas, Brad Renfro, Rachael ...",7,Peter Hewitt
8,9091,"[Jean-Claude Van Damme, Powers Boothe, Dorian ...",6,Peter Hyams
9,710,"[Pierce Brosnan, Sean Bean, Izabella Scorupco]",20,Martin Campbell


In [16]:
#проверим дубли
credits[credits.duplicated("id", keep=False)].sort_values("id")[["id", "cast_list", "cast_size", "director"]].head(20)

,id,cast_list,cast_size,director
25950,3057,"[Luke Goss, Alec Newman, Julie Delpy]",9,Kevin Connor
25885,3057,"[Luke Goss, Alec Newman, Julie Delpy]",9,Kevin Connor
5865,4912,"[Sam Rockwell, Drew Barrymore, Julia Roberts]",26,George Clooney
33838,4912,"[Sam Rockwell, Drew Barrymore, Julia Roberts]",26,George Clooney
9165,5511,"[Alain Delon, François Périer, Nathalie Delon]",27,Jean-Pierre Melville
7345,5511,"[Alain Delon, François Périer, Nathalie Delon]",27,Jean-Pierre Melville
25969,8767,"[Jean-Paul Belmondo, Raquel Welch, Dany Saval]",10,Claude Zidi
25895,8767,"[Jean-Paul Belmondo, Raquel Welch, Dany Saval]",10,Claude Zidi
25893,9755,"[Joe Lando, Dominic Zamprogna, Natassia Malthe]",15,Matthew Hastings
25967,9755,"[Joe Lando, Dominic Zamprogna, Natassia Malthe]",15,Matthew Hastings


In [17]:
#видим, что дубли чисто технические, поэтому удаляем их
credits = credits.drop_duplicates(subset="id", keep="first").copy()

print("Дубликаты id в credits после очистки:", credits["id"].duplicated().sum())
print("Размер credits после удаления дублей:", credits.shape)

Дубликаты id в credits после очистки: 0
Размер credits после удаления дублей: (45432, 6)


In [18]:
#собираем credits_clean
credits_clean = credits[["id", "cast_list", "cast_size", "director"]].copy()
credits_clean = credits_clean.rename(columns={"id": "id_num"})
credits_clean.head()

,id_num,cast_list,cast_size,director
0,862,"[Tom Hanks, Tim Allen, Don Rickles]",13,John Lasseter
1,8844,"[Robin Williams, Jonathan Hyde, Kirsten Dunst]",26,Joe Johnston
2,15602,"[Walter Matthau, Jack Lemmon, Ann-Margret]",7,Howard Deutch
3,31357,"[Whitney Houston, Angela Bassett, Loretta Devine]",10,Forest Whitaker
4,11862,"[Steve Martin, Diane Keaton, Martin Short]",12,Charles Shyer


----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# 4) распарсим keywords

In [19]:
#используем функцию extract_name_list, потому что нам точно так же нужно достать name
keywords["keywords_list"] = keywords["keywords"].apply(extract_name_list)
keywords["keyword_count"] = keywords["keywords_list"].apply(len)

In [20]:
#проверим
keywords[["id", "keywords_list", "keyword_count"]].head(10)

,id,keywords_list,keyword_count
0,862,"[jealousy, toy, boy, friendship, friends, riva...",9
1,8844,"[board game, disappearance, based on children'...",6
2,15602,"[fishing, best friend, duringcreditsstinger, o...",4
3,31357,"[based on novel, interracial relationship, sin...",5
4,11862,"[baby, midlife crisis, confidence, aging, daug...",9
5,949,"[robbery, detective, bank, obsession, chase, s...",25
6,11860,"[paris, brother brother relationship, chauffeu...",6
7,45325,[],0
8,9091,"[terrorist, hostage, explosive, vice president]",4
9,710,"[cuba, falsely accused, secret identity, compu...",15


In [21]:
#проверим дубли
keywords[keywords.duplicated("id", keep=False)].sort_values("id")[["id", "keywords_list", "keyword_count"]].head(20)

,id,keywords_list,keyword_count
37095,1998,"[corruption, hotel, bestechung, adoption, camb...",8
36138,1998,"[corruption, hotel, bestechung, adoption, camb...",8
36822,3025,"[london england, double life, hammer horror, j...",4
35865,3025,"[london england, double life, hammer horror, j...",4
35999,3692,"[spy, cia, eurospy]",3
36956,3692,"[spy, cia, eurospy]",3
35674,4459,"[composer, liquor, flush]",3
36631,4459,"[composer, liquor, flush]",3
36894,4709,[],0
35937,4709,[],0


In [22]:
#опять дубли чисто технические, поэтому удаляем их
keywords = keywords.drop_duplicates(subset="id", keep="first").copy()

print("Дубликаты id в keywords после очистки:", keywords["id"].duplicated().sum())
print("Размер keywords после удаления дублей:", keywords.shape)

Дубликаты id в keywords после очистки: 0
Размер keywords после удаления дублей: (45432, 4)


In [23]:
#собираем keywords_clean
keywords_clean = keywords[["id", "keywords_list", "keyword_count"]].copy()
keywords_clean = keywords_clean.rename(columns={"id": "id_num"})
keywords_clean.head()

,id_num,keywords_list,keyword_count
0,862,"[jealousy, toy, boy, friendship, friends, riva...",9
1,8844,"[board game, disappearance, based on children'...",6
2,15602,"[fishing, best friend, duringcreditsstinger, o...",4
3,31357,"[based on novel, interracial relationship, sin...",5
4,11862,"[baby, midlife crisis, confidence, aging, daug...",9


----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# 5) ratings_summary

по каждому фильму: средняя пользовательская оценка, количество оценок

In [24]:
ratings_summary = (ratings.groupby("movieId", as_index=False).agg(
        avg_rating=("rating", "mean"),
        rating_count=("rating", "count")
    )
)


print(ratings_summary.head(10))
print(ratings_summary.shape)
print(ratings_summary["movieId"].nunique())

   movieId  avg_rating  rating_count
0        1    3.872470           247
1        2    3.401869           107
2        3    3.161017            59
3        4    2.384615            13
4        5    3.267857            56
5        6    3.884615           104
6        7    3.283019            53
7        8    3.800000             5
8        9    3.150000            20
9       10    3.450820           122
(9066, 3)
9066


----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# 6) links_clean

нужно связать ratings_small.movieId с movies.id_num, для этого используем links_small

In [25]:
# В links_small:
# movieId — это ключ из ratings_small
# tmdbId — это ключ, который должен совпасть с movies.id_num

#Но tmdbId был float и с пропусками, поэтому мы:
#приводим его к числу
#выбрасываем строки, где tmdbId отсутствует
#переводим в int64

links["tmdbId"] = pd.to_numeric(links["tmdbId"], errors="coerce")
links_clean = links[links["tmdbId"].notna()].copy()
links_clean["tmdbId"] = links_clean["tmdbId"].astype("int64")

In [26]:
#проверим
links_clean.head()
links_clean.info()

<class 'pandas.DataFrame'>
Index: 9112 entries, 0 to 9124
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9112 non-null   int64
 1   imdbId   9112 non-null   int64
 2   tmdbId   9112 non-null   int64
dtypes: int64(3)
memory usage: 284.8 KB


----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# 7) соединяем рейтинги с tmdbId

In [27]:
ratings_enriched = ratings_summary.merge(
    links_clean[["movieId", "tmdbId"]],
    on="movieId",
    how="left"
)

In [28]:
#проверим
ratings_enriched.head(10)
ratings_enriched.isna().sum()

movieId          0
avg_rating       0
rating_count     0
tmdbId          13
dtype: int64

----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# 8) собираем movies_clean

In [29]:
movies_clean = movies[[
    "id_num",
    "title",
    "original_title",
    "original_language",
    "release_date",
    "release_year",
    "budget",
    "revenue",
    "runtime",
    "popularity",
    "vote_average",
    "vote_count",
    "genres_list",
    "genre_count",
    "main_genre"
]].copy()


movies_clean.head()

,id_num,title,original_title,original_language,release_date,release_year,budget,revenue,runtime,popularity,vote_average,vote_count,genres_list,genre_count,main_genre
0,862,Toy Story,Toy Story,en,1995-10-30,1995.0,30000000,373554033.0,81.0,21.946943,7.7,5415.0,"[Animation, Comedy, Family]",3,Animation
1,8844,Jumanji,Jumanji,en,1995-12-15,1995.0,65000000,262797249.0,104.0,17.015539,6.9,2413.0,"[Adventure, Fantasy, Family]",3,Adventure
2,15602,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,1995.0,0,0.0,101.0,11.712900,6.5,92.0,"[Romance, Comedy]",2,Romance
3,31357,Waiting to Exhale,Waiting to Exhale,en,1995-12-22,1995.0,16000000,81452156.0,127.0,3.859495,6.1,34.0,"[Comedy, Drama, Romance]",3,Comedy
4,11862,Father of the Bride Part II,Father of the Bride Part II,en,1995-02-10,1995.0,0,76578911.0,106.0,8.387519,5.7,173.0,[Comedy],1,Comedy


----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# 9) собираем movie_base

In [30]:
movie_base = (
    movies_clean
    .merge(credits_clean, on="id_num", how="left")
    .merge(keywords_clean, on="id_num", how="left")
    .merge(ratings_enriched, left_on="id_num", right_on="tmdbId", how="left")
)


movie_base.head()
movie_base.shape
movie_base.columns

Index(['id_num', 'title', 'original_title', 'original_language',
       'release_date', 'release_year', 'budget', 'revenue', 'runtime',
       'popularity', 'vote_average', 'vote_count', 'genres_list',
       'genre_count', 'main_genre', 'cast_list', 'cast_size', 'director',
       'keywords_list', 'keyword_count', 'movieId', 'avg_rating',
       'rating_count', 'tmdbId'],
      dtype='str')

----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# 10) посчитаем полезные признаки


In [31]:
#в датасете много фильмов, где бюджет не указан или равен 0
#нам важно отделить фильмы с реальными финансовыми данными от фильмов без них
movie_base["has_budget"] = movie_base["budget"].fillna(0).gt(0).astype(int)
#то же самое с выручкой
movie_base["has_revenue"] = movie_base["revenue"].fillna(0).gt(0).astype(int)
movie_base["profit"] = movie_base["revenue"] - movie_base["budget"]
#roi = revenue / budget
movie_base["roi"] = np.where(
    movie_base["budget"] > 0,
    movie_base["revenue"] / movie_base["budget"],
    np.nan
)

----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# 11) посчитаем взвешенный рейтинг

In [32]:
#учитывает и среднюю оценку фильма, и количество оценок
C = movie_base["avg_rating"].mean() #средний рейтинг по всем фильмам
m = movie_base["rating_count"].quantile(0.75) #такое количество оценок, выше которого находится только 25% фильмов

movie_base["weighted_rating"] = np.where(
    movie_base["rating_count"].notna(),
    (movie_base["rating_count"] / (movie_base["rating_count"] + m)) * movie_base["avg_rating"] +
    (m / (movie_base["rating_count"] + m)) * C,
    np.nan
)

In [33]:
movie_base.head(10)

,id_num,title,original_title,original_language,release_date,release_year,budget,revenue,runtime,popularity,...,keyword_count,movieId,avg_rating,rating_count,tmdbId,has_budget,has_revenue,profit,roi,weighted_rating
0,862,Toy Story,Toy Story,en,1995-10-30,1995.0,30000000,373554033.0,81.0,21.946943,...,9.0,1.0,3.872470,247.0,862.0,1,1,343554033.0,12.451801,3.851999
1,8844,Jumanji,Jumanji,en,1995-12-15,1995.0,65000000,262797249.0,104.0,17.015539,...,6.0,2.0,3.401869,107.0,8844.0,1,1,197797249.0,4.043035,3.393205
2,15602,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,1995.0,0,0.0,101.0,11.712900,...,4.0,3.0,3.161017,59.0,15602.0,0,0,0.0,NaN,3.178115
3,31357,Waiting to Exhale,Waiting to Exhale,en,1995-12-22,1995.0,16000000,81452156.0,127.0,3.859495,...,5.0,4.0,2.384615,13.0,31357.0,1,1,65452156.0,5.090760,2.755083
4,11862,Father of the Bride Part II,Father of the Bride Part II,en,1995-02-10,1995.0,0,76578911.0,106.0,8.387519,...,9.0,5.0,3.267857,56.0,11862.0,0,1,76578911.0,NaN,3.270951
5,949,Heat,Heat,en,1995-12-15,1995.0,60000000,187436818.0,170.0,17.924927,...,25.0,6.0,3.884615,104.0,949.0,1,1,127436818.0,3.123947,3.837273
6,11860,Sabrina,Sabrina,en,1995-12-15,1995.0,58000000,0.0,127.0,6.677277,...,6.0,7.0,3.283019,53.0,11860.0,1,0,-58000000.0,0.000000,3.284062
7,45325,Tom and Huck,Tom and Huck,en,1995-12-22,1995.0,0,0.0,97.0,2.561161,...,0.0,8.0,3.800000,5.0,45325.0,0,0,0.0,NaN,3.472274
8,9091,Sudden Death,Sudden Death,en,1995-12-22,1995.0,35000000,64350171.0,106.0,5.231580,...,4.0,9.0,3.150000,20.0,9091.0,1,1,29350171.0,1.838576,3.193511
9,710,GoldenEye,GoldenEye,en,1995-11-16,1995.0,58000000,352194034.0,130.0,14.686036,...,15.0,10.0,3.450820,122.0,710.0,1,1,294194034.0,6.072311,3.439785


----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# 12) посчитаем high_rating_flag (что влияет на высокий рейтинг фильма) 

In [34]:
#фильмы без weighted_rating тоже получат 0,
#поэтому дальше их нужно отдельно учитывать как No Rating
threshold = movie_base["weighted_rating"].quantile(0.75)
movie_base["high_rating_flag"] = (movie_base["weighted_rating"] >= threshold).astype(int)

movie_base["high_rating_flag"].value_counts(dropna=False)

high_rating_flag
0    43176
1     2257
Name: count, dtype: int64

----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# 13) финальная проверка

In [35]:
movie_base[[
    "id_num", "title", "release_year", "main_genre",
    "director", "cast_size", "keyword_count",
    "avg_rating", "rating_count", "weighted_rating", "high_rating_flag"
]].head(15)

,id_num,title,release_year,main_genre,director,cast_size,keyword_count,avg_rating,rating_count,weighted_rating,high_rating_flag
0,862,Toy Story,1995.0,Animation,John Lasseter,13.0,9.0,3.872470,247.0,3.851999,1
1,8844,Jumanji,1995.0,Adventure,Joe Johnston,26.0,6.0,3.401869,107.0,3.393205,0
2,15602,Grumpier Old Men,1995.0,Romance,Howard Deutch,7.0,4.0,3.161017,59.0,3.178115,0
3,31357,Waiting to Exhale,1995.0,Comedy,Forest Whitaker,10.0,5.0,2.384615,13.0,2.755083,0
4,11862,Father of the Bride Part II,1995.0,Comedy,Charles Shyer,12.0,9.0,3.267857,56.0,3.270951,0
5,949,Heat,1995.0,Action,Michael Mann,65.0,25.0,3.884615,104.0,3.837273,1
6,11860,Sabrina,1995.0,Comedy,Sydney Pollack,57.0,6.0,3.283019,53.0,3.284062,0
7,45325,Tom and Huck,1995.0,Action,Peter Hewitt,7.0,0.0,3.800000,5.0,3.472274,1
8,9091,Sudden Death,1995.0,Action,Peter Hyams,6.0,4.0,3.150000,20.0,3.193511,0
9,710,GoldenEye,1995.0,Adventure,Martin Campbell,20.0,15.0,3.450820,122.0,3.439785,0


In [36]:
movie_base.info()

<class 'pandas.DataFrame'>
RangeIndex: 45433 entries, 0 to 45432
Data columns (total 30 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   id_num             45433 non-null  int64         
 1   title              45430 non-null  str           
 2   original_title     45433 non-null  str           
 3   original_language  45422 non-null  str           
 4   release_date       45346 non-null  datetime64[us]
 5   release_year       45346 non-null  float64       
 6   budget             45433 non-null  int64         
 7   revenue            45430 non-null  float64       
 8   runtime            45173 non-null  float64       
 9   popularity         45430 non-null  float64       
 10  vote_average       45430 non-null  float64       
 11  vote_count         45430 non-null  float64       
 12  genres_list        45433 non-null  object        
 13  genre_count        45433 non-null  int64         
 14  main_genre       

----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# 14) сохраняем очищенные таблицы

In [37]:
movies_clean.to_csv(processed_dir / "movies_clean.csv", index=False)
credits_clean.to_csv(processed_dir / "credits_clean.csv", index=False)
keywords_clean.to_csv(processed_dir / "keywords_clean.csv", index=False)
ratings_summary.to_csv(processed_dir / "ratings_summary.csv", index=False)
links_clean.to_csv(processed_dir / "links_clean.csv", index=False)
movie_base.to_csv(processed_dir / "movie_base.csv", index=False)

print("Все очищенные таблицы сохранены")

Все очищенные таблицы сохранены


---

# **вывод по ноутбуку:**

В таблице `movies_metadata` удалили строки с некорректным `id`, создали числовой идентификатор `id_num`, привели к числовому формату `budget`, `revenue`, `runtime`, `popularity`, `vote_average` и `vote_count`, а также преобразовали дату релиза и выделили год выпуска фильма. После удаления технических дублей итоговая таблица фильмов содержит 45 433 строки.

Из `genres` получены список жанров, количество жанров и основной жанр фильма. Из `credits` получили список актёров, размер актёрского состава и режиссёр. Из `keywords` получили список ключевых слов и их количество. В таблицах `credits` и `keywords` также удалили технические дубли по `id`.

На основе `ratings_small` создали агрегированную таблицу `ratings_summary`, где для каждого фильма были рассчитаны средняя пользовательская оценка и количество оценок. Всего в ней получилось 9 066 фильмов с пользовательскими рейтингами. Затем рейтинги были связаны с TMDB-идентификаторами через `links_small`, после чего стало возможно объединить пользовательские оценки с метаданными фильмов.

В результате была собрана основная аналитическая таблица `movie_base`, в которую вошли метаданные фильмов, жанры, режиссёры, актёрский состав, ключевые слова, пользовательские рейтинги и дополнительные признаки. Финальная таблица содержит 45 433 строки и 30 столбцов.

Дополнительно раcсчитали полезные признаки для дальнейшего анализа:

- `has_budget` — есть ли у фильма ненулевой бюджет
- `has_revenue` — есть ли у фильма ненулевая выручка
- `profit` — разница между выручкой и бюджетом
- `roi` — показатель окупаемости
- `weighted_rating` — взвешенный пользовательский рейтинг
- `high_rating_flag` — флаг высокого рейтинга

Для определения высокорейтинговых фильмов использовался верхний квартиль `weighted_rating`. В результате 2 257 фильмов получили значение `high_rating_flag = 1`, а остальные 43 176 фильмов — `high_rating_flag = 0`
 
Важно! пользовательские рейтинги есть не для всех фильмов (`weighted_rating`, `avg_rating` и `rating_count` заполнены только для 9 025 фильмов). Поэтому на следующих этапах анализ факторов высокого рейтинга нужно проводить именно по фильмам с непустым рейтингом, чтобы не смешивать фильмы без оценок с фильмами низкого качества.

В конце сохранили очищенные таблицы в папку `data/processed`: `movies_clean.csv`, `credits_clean.csv`, `keywords_clean.csv`, `ratings_summary.csv`, `links_clean.csv` и `movie_base.csv`.

Следующий этап проекта — `03_postgres_load.ipynb`, где очищенные данные будут загружены в PostgreSQL, а затем на их основе будут созданы аналитические витрины в схеме `marts`.